# AUT Concept-Bucket Annotation Notebook

This notebook prepares pooled AUT input files for the EMNLP-style concept-bucketing pipeline and then runs the original annotator utilities.

The goal is to assign both human and AI AUT responses to object-specific conceptual buckets using the same pipeline.

For each AUT object, we pool:

- human AUT ideas;
- OpenAI AUT ideas;
- Claude AUT ideas;
- Gemini AUT ideas;
- main and robustness AI generations.

Each object gets its own codebook because bucket IDs are only meaningful within an object condition.

## 1. Imports and paths

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Any, Optional
import os
import sys
import json
import shutil
import hashlib
import datetime as dt

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 180)

BASE_DIR = Path(".")
ANALYSIS_DIR = Path("analysis_outputs")
STANDARDIZED_DIR = ANALYSIS_DIR / "standardized_loaded_data"
AI_PROCESSED_DIR = Path("ai_data/processed")

AUT_BUCKET_DIR = ANALYSIS_DIR / "aut_bucket_annotation"
INPUT_FILES_DIR = Path("input_files")

AUT_BUCKET_DIR.mkdir(parents=True, exist_ok=True)
INPUT_FILES_DIR.mkdir(parents=True, exist_ok=True)

POOLED_INPUT_DIR = AUT_BUCKET_DIR / "pooled_input_csvs"
METADATA_DIR = AUT_BUCKET_DIR / "metadata"
EXPORT_COPY_DIR = AUT_BUCKET_DIR / "exports_copy"

for p in [POOLED_INPUT_DIR, METADATA_DIR, EXPORT_COPY_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Working directory:", Path.cwd())
print("AUT bucket output directory:", AUT_BUCKET_DIR)

Working directory: C:\Users\Raiyan Abdul Baten\Dropbox\py_virtual_env_dec20\nafis_creativity\neurips_analysis
AUT bucket output directory: analysis_outputs\aut_bucket_annotation


## 2. Helper functions

We use stable IDs so that repeated notebook runs produce the same row IDs unless the underlying data change.

In [2]:
def stable_hash(obj: Any) -> str:
    payload = json.dumps(obj, sort_keys=True, ensure_ascii=False, default=str)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def short_hash(obj: Any, n: int = 16) -> str:
    return stable_hash(obj)[:n]


def normalize_text_for_id(text: str) -> str:
    if not isinstance(text, str):
        return ""
    return " ".join(text.strip().split())


def clean_object_name(x: str) -> str:
    """
    Convert object names to the exact object_name strings expected by forbidden_ideas.csv.
    """
    x = str(x).strip().lower()
    x = x.replace("-", " ")
    x = " ".join(x.split())

    mapping = {
        "shoe": "shoe",
        "button": "button",
        "key": "key",
        "wooden pencil": "wooden_pencil",
        "wooden_pencil": "wooden_pencil",
        "automobile tire": "automobile_tire",
        "automobile_tire": "automobile_tire",
        "tire": "automobile_tire",
    }

    if x not in mapping:
        raise ValueError(f"Unknown AUT object name: {x}")

    return mapping[x]


AUT_OBJECT_ORDER = ["shoe", "button", "key", "wooden_pencil", "automobile_tire"]

OBJECT_TO_ROUND_ID = {
    "shoe": 1,
    "button": 2,
    "key": 3,
    "wooden_pencil": 4,
    "automobile_tire": 5,
}

OBJECT_DISPLAY = {
    "shoe": "shoe",
    "button": "button",
    "key": "key",
    "wooden_pencil": "wooden pencil",
    "automobile_tire": "automobile tire",
}

COMMON_USE = {
    "shoe": "used as footwear",
    "button": "used to fasten things",
    "key": "used to open a lock",
    "wooden_pencil": "used for writing",
    "automobile_tire": "used on wheel of automobile",
}

## 3. Create `forbidden_ideas.csv`

The original utility file looks for this file at:

`input_files/forbidden_ideas.csv`

The `forbidden_idea` is the common/forbidden use of the object and is inserted as comparison idea ID `0` by the original pipeline.

In [3]:
forbidden_ideas_df = pd.DataFrame(
    [
        {
            "object_name": obj,
            "forbidden_idea": COMMON_USE[obj],
        }
        for obj in AUT_OBJECT_ORDER
    ]
)

forbidden_path = INPUT_FILES_DIR / "forbidden_ideas.csv"
forbidden_ideas_df.to_csv(forbidden_path, index=False)

print("Saved:", forbidden_path)
display(forbidden_ideas_df)

Saved: input_files\forbidden_ideas.csv


,object_name,forbidden_idea
0,shoe,used as footwear
1,button,used to fasten things
2,key,used to open a lock
3,wooden_pencil,used for writing
4,automobile_tire,used on wheel of automobile


## 4. Load human AUT data

We load the curated human AUT data from the standardized analysis outputs. This should be the full human response table, not the sampled one-response version.

In [4]:
human_standard_path = STANDARDIZED_DIR / "human_standard_all_tasks.pkl"
assert human_standard_path.exists(), f"Missing: {human_standard_path}"

human_standard_df = pd.read_pickle(human_standard_path)

human_aut_df = human_standard_df.query("task_family == 'aut'").copy()

print("Human AUT rows:", human_aut_df.shape)
display(human_aut_df.head())

display(
    human_aut_df
    .groupby(["condition_id", "prompt_or_object"], dropna=False)
    .agg(
        n_ideas=("response_text", "size"),
        n_participants=("participant_id", "nunique"),
    )
    .reset_index()
)

Human AUT rows: (3047, 17)


,task_family,source_type,participant_id,condition_id,condition_label,prompt,response_text,response_word_count,response_char_count,response_sentence_count,prompt_or_object,bucket_id,bucket_key,order,common_use,valid_idea_heuristic,valid_slogan_heuristic
87,aut,human,100,shoe,Shoe,NaN,Swatting Flies,2,14,1,shoe,18,shoe::18,3699,used as footwear,True,NaN
88,aut,human,100,shoe,Shoe,NaN,Used as a hat,4,13,1,shoe,177,shoe::177,3700,used as footwear,True,NaN
89,aut,human,100,shoe,Shoe,NaN,Used as a pot for a plant,7,25,1,shoe,266,shoe::266,3701,used as footwear,True,NaN
90,aut,human,100,shoe,Shoe,NaN,Contain dripping leaks,3,22,1,shoe,133,shoe::133,3702,used as footwear,True,NaN
91,aut,human,100,shoe,Shoe,NaN,Drumming,1,8,1,shoe,371,shoe::371,3703,used as footwear,True,NaN


,condition_id,prompt_or_object,n_ideas,n_participants
0,automobile_tire,automobile tire,615,109
1,button,button,603,109
2,key,key,612,109
3,shoe,shoe,604,109
4,wooden_pencil,wooden pencil,613,109


## 5. Standardize human AUT rows for pooled annotation

The original bucketing utility requires at least:

- `id`
- `for_user_id`
- `round_id`
- `idea_content`

We create a new stable integer `id` for this pooled annotation project and store the original metadata separately.

In [5]:
def standardize_human_aut_for_bucketing(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Infer object name from prompt_or_object when available.
    out["object_name"] = out["prompt_or_object"].map(clean_object_name)

    out["idea_content"] = out["response_text"].astype(str).map(normalize_text_for_id)
    out = out[out["idea_content"].str.len() > 0].copy()

    out["source_group"] = "human"
    out["provider"] = "human"
    out["model"] = "human"
    out["analysis_scenario_name"] = "human"
    out["scenario_name"] = "human"
    out["temperature"] = np.nan
    out["persona_id"] = np.nan

    out["for_user_id"] = "human_" + out["participant_id"].astype(str)
    out["round_id"] = out["object_name"].map(OBJECT_TO_ROUND_ID)

    # Preserve available manual annotation for later optional comparison only.
    if "bucket_key" not in out.columns:
        out["bucket_key"] = np.nan
    if "bucket_id" not in out.columns:
        out["bucket_id"] = np.nan

    keep = [
        "source_group",
        "provider",
        "model",
        "analysis_scenario_name",
        "scenario_name",
        "temperature",
        "persona_id",
        "object_name",
        "round_id",
        "for_user_id",
        "participant_id",
        "idea_content",
        "bucket_id",
        "bucket_key",
    ]

    out = out[keep].copy()
    return out


human_aut_bucket_df = standardize_human_aut_for_bucketing(human_aut_df)

print("Standardized human AUT rows:", human_aut_bucket_df.shape)

display(
    human_aut_bucket_df
    .groupby("object_name")
    .agg(
        n=("idea_content", "size"),
        n_users=("for_user_id", "nunique"),
        n_unique_texts=("idea_content", "nunique"),
    )
    .reset_index()
)

human_aut_bucket_df.head()

Standardized human AUT rows: (3047, 14)


,object_name,n,n_users,n_unique_texts
0,automobile_tire,615,109,575
1,button,603,109,583
2,key,612,109,574
3,shoe,604,109,532
4,wooden_pencil,613,109,583


,source_group,provider,model,analysis_scenario_name,scenario_name,temperature,persona_id,object_name,round_id,for_user_id,participant_id,idea_content,bucket_id,bucket_key
87,human,human,human,human,human,NaN,NaN,shoe,1,human_100,100,Swatting Flies,18,shoe::18
88,human,human,human,human,human,NaN,NaN,shoe,1,human_100,100,Used as a hat,177,shoe::177
89,human,human,human,human,human,NaN,NaN,shoe,1,human_100,100,Used as a pot for a plant,266,shoe::266
90,human,human,human,human,human,NaN,NaN,shoe,1,human_100,100,Contain dripping leaks,133,shoe::133
91,human,human,human,human,human,NaN,NaN,shoe,1,human_100,100,Drumming,371,shoe::371


## 6. Load all AI AUT data

We load the success-only AUT generation files for all three providers. These include the main condition and robustness variants.

In [6]:
AI_AUT_FILES = [
    ("openai", "gpt-5.4", AI_PROCESSED_DIR / "openai__gpt-5.4__aut_generations_success_only.pkl"),
    ("anthropic", "claude-sonnet-4-5", AI_PROCESSED_DIR / "anthropic__claude-sonnet-4-5__aut_generations_success_only.pkl"),
    ("gemini", "gemini-2.5-flash", AI_PROCESSED_DIR / "gemini__gemini-2.5-flash__aut_generations_success_only.pkl"),
]

ai_aut_dfs = []

for provider, model, path in AI_AUT_FILES:
    assert path.exists(), f"Missing: {path}"

    df = pd.read_pickle(path).copy()
    df["provider"] = provider
    df["model"] = model
    df["source_group"] = "ai"
    df["task_family"] = "aut"

    if "analysis_scenario_name" not in df.columns:
        df["analysis_scenario_name"] = df.get("scenario_name", np.nan)

    df["analysis_scenario_name"] = df["analysis_scenario_name"].fillna(df.get("scenario_name", np.nan))

    ai_aut_dfs.append(df)

    print(f"Loaded {provider} / {model}: {df.shape}")

ai_aut_df = pd.concat(ai_aut_dfs, ignore_index=True, sort=False)

print("All AI AUT rows:", ai_aut_df.shape)
display(ai_aut_df.head())

display(
    ai_aut_df
    .groupby(["provider", "model", "analysis_scenario_name", "temperature"], dropna=False)
    .size()
    .reset_index(name="n")
    .sort_values(["provider", "analysis_scenario_name", "temperature"])
)

Loaded openai / gpt-5.4: (5150, 35)
Loaded anthropic / claude-sonnet-4-5: (5150, 37)
Loaded gemini / gemini-2.5-flash: (5150, 37)
All AI AUT rows: (15450, 39)


,request_key,task_family,scenario_name,provider,model,condition_type,condition_id,condition_label,object,common_use,temperature,run_idx,persona_id,persona_traits,system_instructions,user_prompt,max_output_tokens,created_at_utc,status,text,provider_response_id,usage,raw_response,error,batch_custom_id,batch_output_file,source_file,output_word_count,output_line_count,looks_like_list,likely_multiple_uses,repeats_primary_use_terms,valid_exactly_one_short_response_heuristic,source_group,analysis_scenario_name,anthropic_custom_id,batch_id,gemini_custom_id,batch_name
0,000a5fd26b8631398da5335fe6bea3af486c2a9e5ec33e6814da8859e36c8248,aut,personality_grid,openai,gpt-5.4,personality,button,Button,button,used to fasten things,1.3,8,extroverted__agreeable__conscientious__emotionally_stable__closed_to_experience,"['extroverted', 'agreeable', 'conscientious', 'emotionally_stable', 'closed to experience']",You are participating in a creativity test. Follow the task instructions exactly. Do not explain your reasoning. Do not include commentary before or after the answer.\n\nWrite ...,"Object: button\nCommon use to avoid: used to fasten things\n\nGenerate exactly one unusual, creative, and plausible alternative use for the object or one of its parts. Do not u...",120,2026-04-29T16:46:01.221875+00:00,success,Use a button as a neat circular paint stamp for making uniform polka-dot patterns.,resp_07019e4b3e4d863c0069f22d4e36e8819795fcfb2b0e422559,"{'input_tokens': 153, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 21, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 174}",None,None,000a5fd26b8631398da5335fe6bea3af486c2a9e5ec33e6814da8859e36c8248,ai_data/aut_batches/openai/batch_69f22bf131088190a37c091663450f08__output.jsonl,ai_data/aut_generations/openai__gpt-5.4__personality_grid.jsonl,15,1,False,False,False,True,ai,personality_grid,NaN,NaN,NaN,NaN
1,000d7acf77d67670e15297dc3861b8cad2507446f80c70a6f0b5e9d248aed610,aut,personality_grid,openai,gpt-5.4,personality,shoe,Shoe,shoe,used as footwear,0.7,0,introverted__antagonistic__unconscientious__emotionally_stable__open_to_experience,"['introverted', 'antagonistic', 'unconscientious', 'emotionally_stable', 'open to experience']",You are participating in a creativity test. Follow the task instructions exactly. Do not explain your reasoning. Do not include commentary before or after the answer.\n\nWrite ...,"Object: shoe\nCommon use to avoid: used as footwear\n\nGenerate exactly one unusual, creative, and plausible alternative use for the object or one of its parts. Do not use the ...",120,2026-04-29T16:46:01.425813+00:00,success,A shoe can be hung by its laces as a discreet windowsill planter for a trailing vine.,resp_0a5c23a3bf98f7410069f22e3789d88197b5084bd7ff916359,"{'input_tokens': 153, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 24, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 177}",None,None,000d7acf77d67670e15297dc3861b8cad2507446f80c70a6f0b5e9d248aed610,ai_data/aut_batches/openai/batch_69f22bf131088190a37c091663450f08__output.jsonl,ai_data/aut_generations/openai__gpt-5.4__personality_grid.jsonl,17,1,False,False,False,True,ai,personality_grid,NaN,NaN,NaN,NaN
2,00138119accfc2d6aabda6991f2ab53675a808d0e8791ad750a5077679a76808,aut,personality_grid,openai,gpt-5.4,personality,shoe,Shoe,shoe,used as footwear,1.3,2,extroverted__antagonistic__unconscientious__emotionally_stable__open_to_experience,"['extroverted', 'antagonistic', 'unconscientious', 'emotionally_stable', 'open to experience']",You are participating in a creativity test. Follow the task instructions exactly. Do not explain your reasoning. Do not include commentary before or after the answer.\n\nWrite ...,"Object: shoe\nCommon use to avoid: used as footwear\n\nGenerate exactly one unusual, creative, and plausible alternative use for the object or one of its parts. Do not use the ...",120,2026-04-29T16:46:01.495719+00:00,success,Use the shoe as a wedge to prop a stub

,provider,model,analysis_scenario_name,temperature,n
0,anthropic,claude-sonnet-4-5,neutral_main_t1,1.0,250
1,anthropic,claude-sonnet-4-5,neutral_temperature_robustness,0.3,50
2,anthropic,claude-sonnet-4-5,neutral_temperature_robustness,0.7,50
3,anthropic,claude-sonnet-4-5,personality_grid,0.3,1600
4,anthropic,claude-sonnet-4-5,personality_grid,0.7,1600
5,anthropic,claude-sonnet-4-5,personality_grid,1.0,1600
6,gemini,gemini-2.5-flash,neutral_main_t1,1.0,250
7,gemini,gemini-2.5-flash,neutral_temperature_robustness,0.7,50
8,gemini,gemini-2.5-flash,neutral_temperature_robustness,1.3,50
9,gemini,gemini-2.5-flash,personality_grid,0.7,1600


## 7. Standardize AI AUT rows for pooled annotation

We create a synthetic `for_user_id` for each AI generation attempt. This ID is not used by the bucketing utility except as metadata, but it makes the exported annotation file easier to merge later.

In [7]:
def infer_ai_aut_object_name(row: pd.Series) -> str:
    """
    Best-effort object-name inference from AI AUT generation rows.
    """
    for col in ["object_name", "object", "prompt_or_object", "condition_label", "condition_id"]:
        if col in row.index:
            val = row.get(col)
            if isinstance(val, str) and val.strip() and val.strip().lower() != "nan":
                try:
                    return clean_object_name(val)
                except Exception:
                    pass

    # Some condition IDs may be numeric round IDs.
    condition_id = row.get("condition_id")
    try:
        rid = int(float(condition_id))
        round_to_object = {v: k for k, v in OBJECT_TO_ROUND_ID.items()}
        if rid in round_to_object:
            return round_to_object[rid]
    except Exception:
        pass

    raise ValueError(f"Could not infer AUT object from row: {row.to_dict()}")


def standardize_ai_aut_for_bucketing(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Standard response text column.
    if "response_text" not in out.columns:
        if "text" in out.columns:
            out["response_text"] = out["text"].astype(str)
        else:
            raise ValueError("AI AUT dataframe has neither response_text nor text.")

    out["idea_content"] = out["response_text"].astype(str).map(normalize_text_for_id)
    out = out[out["idea_content"].str.len() > 0].copy()

    out["object_name"] = out.apply(infer_ai_aut_object_name, axis=1)
    out["round_id"] = out["object_name"].map(OBJECT_TO_ROUND_ID)

    for col in ["scenario_name", "analysis_scenario_name", "temperature", "persona_id", "run_idx", "request_key"]:
        if col not in out.columns:
            out[col] = np.nan

    # Build a stable generation-attempt identifier.
    out["ai_generation_uid"] = out.apply(
        lambda row: short_hash(
            {
                "provider": row["provider"],
                "model": row["model"],
                "analysis_scenario_name": row["analysis_scenario_name"],
                "scenario_name": row["scenario_name"],
                "temperature": None if pd.isna(row["temperature"]) else float(row["temperature"]),
                "persona_id": None if pd.isna(row["persona_id"]) else str(row["persona_id"]),
                "object_name": row["object_name"],
                "run_idx": None if pd.isna(row["run_idx"]) else str(row["run_idx"]),
                "request_key": None if pd.isna(row["request_key"]) else str(row["request_key"]),
                "idea_content": row["idea_content"],
            },
            n=20,
        ),
        axis=1,
    )

    out["for_user_id"] = "ai_" + out["provider"].astype(str) + "_" + out["ai_generation_uid"].astype(str)
    out["participant_id"] = np.nan
    out["bucket_id"] = np.nan
    out["bucket_key"] = np.nan

    keep = [
        "source_group",
        "provider",
        "model",
        "analysis_scenario_name",
        "scenario_name",
        "temperature",
        "persona_id",
        "object_name",
        "round_id",
        "for_user_id",
        "participant_id",
        "idea_content",
        "ai_generation_uid",
        "run_idx",
        "request_key",
        "bucket_id",
        "bucket_key",
    ]

    return out[keep].copy()


ai_aut_bucket_df = standardize_ai_aut_for_bucketing(ai_aut_df)

print("Standardized AI AUT rows:", ai_aut_bucket_df.shape)

display(
    ai_aut_bucket_df
    .groupby(["object_name", "provider", "model", "analysis_scenario_name"], dropna=False)
    .size()
    .reset_index(name="n")
    .sort_values(["object_name", "provider", "analysis_scenario_name"])
)

ai_aut_bucket_df.head()

Standardized AI AUT rows: (15450, 17)


,object_name,provider,model,analysis_scenario_name,n
0,automobile_tire,anthropic,claude-sonnet-4-5,neutral_main_t1,50
1,automobile_tire,anthropic,claude-sonnet-4-5,neutral_temperature_robustness,20
2,automobile_tire,anthropic,claude-sonnet-4-5,personality_grid,960
3,automobile_tire,gemini,gemini-2.5-flash,neutral_main_t1,50
4,automobile_tire,gemini,gemini-2.5-flash,neutral_temperature_robustness,20
5,automobile_tire,gemini,gemini-2.5-flash,personality_grid,960
6,automobile_tire,openai,gpt-5.4,neutral_main_t1,50
7,automobile_tire,openai,gpt-5.4,neutral_temperature_robustness,20
8,automobile_tire,openai,gpt-5.4,personality_grid,960
9,button,anthropic,claude-sonnet-4-5,neutral_main_t1,50


,source_group,provider,model,analysis_scenario_name,scenario_name,temperature,persona_id,object_name,round_id,for_user_id,participant_id,idea_content,ai_generation_uid,run_idx,request_key,bucket_id,bucket_key
0,ai,openai,gpt-5.4,personality_grid,personality_grid,1.3,extroverted__agreeable__conscientious__emotionally_stable__closed_to_experience,button,2,ai_openai_63d98ddde5d71510a156,NaN,Use a button as a neat circular paint stamp for making uniform polka-dot patterns.,63d98ddde5d71510a156,8,000a5fd26b8631398da5335fe6bea3af486c2a9e5ec33e6814da8859e36c8248,NaN,NaN
1,ai,openai,gpt-5.4,personality_grid,personality_grid,0.7,introverted__antagonistic__unconscientious__emotionally_stable__open_to_experience,shoe,1,ai_openai_e27fd7e68ceec4a83616,NaN,A shoe can be hung by its laces as a discreet windowsill planter for a trailing vine.,e27fd7e68ceec4a83616,0,000d7acf77d67670e15297dc3861b8cad2507446f80c70a6f0b5e9d248aed610,NaN,NaN
2,ai,openai,gpt-5.4,personality_grid,personality_grid,1.3,extroverted__antagonistic__unconscientious__emotionally_stable__open_to_experience,shoe,1,ai_openai_2910a950a9e01dd6a2e9,NaN,Use the shoe as a wedge to prop a stubborn door open.,2910a950a9e01dd6a2e9,2,00138119accfc2d6aabda6991f2ab53675a808d0e8791ad750a5077679a76808,NaN,NaN
3,ai,openai,gpt-5.4,personality_grid,personality_grid,1.0,introverted__antagonistic__conscientious__emotionally_stable__closed_to_experience,wooden_pencil,4,ai_openai_8d0acaca28d67be0ccee,NaN,Use the pencil’s ferrule and eraser end as a quiet wedge to keep a rattling window from shifting.,8d0acaca28d67be0ccee,6,00234f452599d9a5898ffdbc52265bcf8415764064884457c2f61936b2c9c43e,NaN,NaN
4,ai,openai,gpt-5.4,personality_grid,personality_grid,1.3,introverted__antagonistic__conscientious__neurotic__open_to_experience,automobile_tire,5,ai_openai_f269988d5af03226d8c0,NaN,Half-bury the tire upright in a playground to create a durable crawl-through tunnel for children.,f269988d5af03226d8c0,7,002889542fd97a6005b63437946da15ad03a808d79e8c14067361fb4cd5b78f3,NaN,NaN


## 8. Pool human and AI AUT rows

We pool human and AI responses within each AUT object. The object-specific CSVs are what the EMNLP bucketing utility will annotate.

Important: exact duplicate ideas are kept if they come from distinct human participants or distinct AI generation attempts. Duplicates are part of the redundancy signal.

In [8]:
aut_pooled_df = pd.concat(
    [human_aut_bucket_df, ai_aut_bucket_df],
    ignore_index=True,
    sort=False,
)

aut_pooled_df["idea_content"] = aut_pooled_df["idea_content"].astype(str).map(normalize_text_for_id)
aut_pooled_df = aut_pooled_df[aut_pooled_df["idea_content"].str.len() > 0].copy()

# Create a stable row UID before assigning integer IDs.
aut_pooled_df["pooled_row_uid"] = aut_pooled_df.apply(
    lambda row: short_hash(
        {
            "source_group": row["source_group"],
            "provider": row["provider"],
            "model": row["model"],
            "analysis_scenario_name": row["analysis_scenario_name"],
            "scenario_name": row["scenario_name"],
            "temperature": None if pd.isna(row["temperature"]) else float(row["temperature"]),
            "persona_id": None if pd.isna(row["persona_id"]) else str(row["persona_id"]),
            "object_name": row["object_name"],
            "for_user_id": row["for_user_id"],
            "idea_content": row["idea_content"],
            "ai_generation_uid": None if pd.isna(row.get("ai_generation_uid", np.nan)) else str(row.get("ai_generation_uid")),
        },
        n=24,
    ),
    axis=1,
)

# Ensure row-level uniqueness. If duplicates remain, add row index.
if not aut_pooled_df["pooled_row_uid"].is_unique:
    print("Found duplicate pooled_row_uid values; adding row index to disambiguate.")
    aut_pooled_df = aut_pooled_df.reset_index(drop=True)
    aut_pooled_df["pooled_row_idx"] = np.arange(len(aut_pooled_df))
    aut_pooled_df["pooled_row_uid"] = aut_pooled_df.apply(
        lambda row: short_hash(
            {
                "previous_uid": row["pooled_row_uid"],
                "pooled_row_idx": int(row["pooled_row_idx"]),
            },
            n=24,
        ),
        axis=1,
    )
else:
    aut_pooled_df = aut_pooled_df.reset_index(drop=True)
    aut_pooled_df["pooled_row_idx"] = np.arange(len(aut_pooled_df))

# The original utility requires integer id values.
# Use deterministic sequential IDs after sorting by object and pooled_row_uid.
aut_pooled_df = (
    aut_pooled_df
    .sort_values(["object_name", "pooled_row_uid"])
    .reset_index(drop=True)
)

aut_pooled_df["id"] = np.arange(1, len(aut_pooled_df) + 1, dtype=int)

print("Pooled AUT rows:", aut_pooled_df.shape)

display(
    aut_pooled_df
    .groupby(["object_name", "source_group"], dropna=False)
    .agg(
        n=("idea_content", "size"),
        n_unique_texts=("idea_content", "nunique"),
        n_users=("for_user_id", "nunique"),
    )
    .reset_index()
    .sort_values(["object_name", "source_group"])
)

aut_pooled_df.head()

C:\Users\Raiyan Abdul Baten\AppData\Local\Temp\ipykernel_38448\3828501518.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  aut_pooled_df = pd.concat(


Pooled AUT rows: (18497, 20)


,object_name,source_group,n,n_unique_texts,n_users
0,automobile_tire,ai,3090,2560,3090
1,automobile_tire,human,615,575,109
2,button,ai,3090,2470,3090
3,button,human,603,583,109
4,key,ai,3090,2635,3090
5,key,human,612,574,109
6,shoe,ai,3090,2171,3090
7,shoe,human,604,532,109
8,wooden_pencil,ai,3090,2422,3090
9,wooden_pencil,human,613,583,109


,source_group,provider,model,analysis_scenario_name,scenario_name,temperature,persona_id,object_name,round_id,for_user_id,participant_id,idea_content,bucket_id,bucket_key,ai_generation_uid,run_idx,request_key,pooled_row_uid,pooled_row_idx,id
0,ai,gemini,gemini-2.5-flash,personality_grid,personality_grid,1.3,extroverted__agreeable__conscientious__emotionally_stable__closed_to_experience,automobile_tire,5,ai_gemini_8c69db3d300850927fef,NaN,"A swing for the kids in the backyard, properly secured and safe.",<NA>,NaN,8c69db3d300850927fef,0.0,0066e0feb14b263c25f7036c1cfce5649438462172533f0b0d086ed27050421e,0008cb215a0a245767472be0,13355,1
1,human,human,human,human,human,NaN,NaN,automobile_tire,5,human_110,110,Dog bed,66,automobile_tire::66,NaN,NaN,NaN,001b9bec7c51aa084564bcec,289,2
2,ai,gemini,gemini-2.5-flash,personality_grid,personality_grid,0.7,introverted__agreeable__unconscientious__emotionally_stable__closed_to_experience,automobile_tire,5,ai_gemini_66ad2b9e3bb368647d05,NaN,A swing for a playground.,<NA>,NaN,66ad2b9e3bb368647d05,2.0,72c6379e83f4f73410fe26859615ee9022d35cdbd1834ac789f453c0569aa4af,00219c865689ae226538c625,15725,3
3,ai,gemini,gemini-2.5-flash,personality_grid,personality_grid,1.3,introverted__agreeable__conscientious__neurotic__closed_to_experience,automobile_tire,5,ai_gemini_45ebc7e9964ae585cb08,NaN,A playground swing base.,<NA>,NaN,45ebc7e9964ae585cb08,3.0,4335bebba66ff562f753b3553c0b556d22c7d2166f09361ed1d55835d483f941,0035032c0e36146f2e1eb9a4,14737,4
4,ai,openai,gpt-5.4,personality_grid,personality_grid,0.7,extroverted__antagonistic__conscientious__neurotic__closed_to_experience,automobile_tire,5,ai_openai_ef35da44c5492562f55d,NaN,Suspend the tire horizontally from a sturdy frame as a rugged livestock salt-lick holder that keeps the block off the muddy ground.,<NA>,NaN,ef35da44c5492562f55d,1.0,2cc76fde5fbdfef1ff8cc4ce2438423a6923835c5c4f9f91dcaf252dd2ed1475,00398a8c27f3fcfaf1ace8c3,3956,5


## 9. Export object-specific input CSVs and metadata

The original utility reads files from `input_files/ideas_<object>.csv`.

We export each object file with the required columns:

- `id`
- `for_user_id`
- `round_id`
- `idea_content`

We also save richer metadata under `analysis_outputs/aut_bucket_annotation/metadata/` for merging annotation results back later.

In [9]:
required_input_cols = ["id", "for_user_id", "round_id", "idea_content"]

metadata_cols = [
    "id",
    "pooled_row_uid",
    "pooled_row_idx",
    "source_group",
    "provider",
    "model",
    "analysis_scenario_name",
    "scenario_name",
    "temperature",
    "persona_id",
    "object_name",
    "round_id",
    "for_user_id",
    "participant_id",
    "idea_content",
    "ai_generation_uid",
    "run_idx",
    "request_key",
    "bucket_id",
    "bucket_key",
]

exported_input_paths = {}

for object_name in AUT_OBJECT_ORDER:
    sub = aut_pooled_df.query("object_name == @object_name").copy()

    input_path = INPUT_FILES_DIR / f"ideas_{object_name}.csv"
    metadata_path = METADATA_DIR / f"pooled_metadata_{object_name}.csv"

    sub[required_input_cols].to_csv(input_path, index=False)
    sub[[c for c in metadata_cols if c in sub.columns]].to_csv(metadata_path, index=False)

    exported_input_paths[object_name] = str(input_path)

    print(f"{object_name:16s} | input rows={len(sub):5d} | {input_path}")
    print(f"{'':16s} | metadata: {metadata_path}")

# Save all pooled metadata too.
all_metadata_path = METADATA_DIR / "pooled_aut_metadata_all_objects.csv"
aut_pooled_df[[c for c in metadata_cols if c in aut_pooled_df.columns]].to_csv(all_metadata_path, index=False)

print("\nSaved all metadata:", all_metadata_path)

shoe             | input rows= 3694 | input_files\ideas_shoe.csv
                 | metadata: analysis_outputs\aut_bucket_annotation\metadata\pooled_metadata_shoe.csv
button           | input rows= 3693 | input_files\ideas_button.csv
                 | metadata: analysis_outputs\aut_bucket_annotation\metadata\pooled_metadata_button.csv
key              | input rows= 3702 | input_files\ideas_key.csv
                 | metadata: analysis_outputs\aut_bucket_annotation\metadata\pooled_metadata_key.csv
wooden_pencil    | input rows= 3703 | input_files\ideas_wooden_pencil.csv
                 | metadata: analysis_outputs\aut_bucket_annotation\metadata\pooled_metadata_wooden_pencil.csv
automobile_tire  | input rows= 3705 | input_files\ideas_automobile_tire.csv
                 | metadata: analysis_outputs\aut_bucket_annotation\metadata\pooled_metadata_automobile_tire.csv

Saved all metadata: analysis_outputs\aut_bucket_annotation\metadata\pooled_aut_metadata_all_objects.csv


## 10. Copy or locate `utils.py`

This notebook uses the original EMNLP utility functions.

Set `UTILS_SOURCE_PATH` to the location of `utils.py` in your local EMNLP project. If the file is already in the current working directory, this cell will import it directly.

In [10]:
# Option 1: if utils.py is already in this notebook's working directory.
UTILS_SOURCE_PATH = Path("utils.py")

# Option 2: if you want to copy from another project, uncomment and edit:
# UTILS_SOURCE_PATH = Path("/path/to/emnlp_project/utils.py")

# Option 3: in this ChatGPT sandbox, the uploaded file was at /mnt/data/utils.py.
# On your machine, replace this with your actual EMNLP path if needed.
if not UTILS_SOURCE_PATH.exists() and Path("/mnt/data/utils.py").exists():
    UTILS_SOURCE_PATH = Path("/mnt/data/utils.py")

assert UTILS_SOURCE_PATH.exists(), f"Could not find utils.py at {UTILS_SOURCE_PATH}"

# Copy to current working directory if needed so `import utils` works.
if UTILS_SOURCE_PATH.resolve() != (Path.cwd() / "utils.py").resolve():
    shutil.copy2(UTILS_SOURCE_PATH, Path("utils.py"))
    print(f"Copied {UTILS_SOURCE_PATH} -> utils.py")
else:
    print("Using local utils.py")

# Import/reload.
import importlib
import utils
importlib.reload(utils)

from utils import run_experiment, export_annotations

print("Loaded utils from:", Path("utils.py").resolve())

Using local utils.py
Loaded utils from: C:\Users\Raiyan Abdul Baten\Dropbox\py_virtual_env_dec20\nafis_creativity\neurips_analysis\utils.py


## 11. Configure bucketing experiments

We use the validated EMNLP-style pipeline:

- LLM judge: `llama3.3:70b` through Ollama;
- embedding model: same as the main NeurIPS semantic analysis;
- comparison candidates: `K_c = 10`;
- prompt method: `CoT`;
- one deterministic object-specific run per object.

Before running this, make sure Ollama is running locally and the model is available:

`ollama pull llama3.3:70b`

In [11]:
LLM_MODEL = "llama3.3:latest"
EMBEDDING_MODEL = "sentence-transformers/all-mpnet-base-v2"
COMPARISON_K = 10
PROMPT_METHOD = "CoT"

# This seed controls the deterministic shuffle used by load_experiment_csv.
# It is not an R=3 replication design; it is a single fixed run.
ANNOTATION_SEED = 20260501

MAX_ATTEMPTS = 3
SAVE_CSVS = True

EXPERIMENT_PREFIX = "neurips_aut_pooled_llama32_70b_k10"

experiment_args_by_object = {}

for object_name in AUT_OBJECT_ORDER:
    exp_name = f"{EXPERIMENT_PREFIX}_{object_name}"

    exp_arg = {
        "experiment_name": exp_name,
        "object_name": object_name,
        "input_csv_path": str(INPUT_FILES_DIR / f"ideas_{object_name}.csv"),
        "llm_model": LLM_MODEL,
        "embedding_model": EMBEDDING_MODEL,
        "comparison_k": COMPARISON_K,
        "prompt_method": PROMPT_METHOD,
        "replication_id": ANNOTATION_SEED,
        "max_attempts": MAX_ATTEMPTS,
        "save_csvs": SAVE_CSVS,
    }

    experiment_args_by_object[object_name] = exp_arg

pd.DataFrame(experiment_args_by_object).T

,experiment_name,object_name,input_csv_path,llm_model,embedding_model,comparison_k,prompt_method,replication_id,max_attempts,save_csvs
shoe,neurips_aut_pooled_llama32_70b_k10_shoe,shoe,input_files\ideas_shoe.csv,llama3.3:latest,sentence-transformers/all-mpnet-base-v2,10,CoT,20260501,3,True
button,neurips_aut_pooled_llama32_70b_k10_button,button,input_files\ideas_button.csv,llama3.3:latest,sentence-transformers/all-mpnet-base-v2,10,CoT,20260501,3,True
key,neurips_aut_pooled_llama32_70b_k10_key,key,input_files\ideas_key.csv,llama3.3:latest,sentence-transformers/all-mpnet-base-v2,10,CoT,20260501,3,True
wooden_pencil,neurips_aut_pooled_llama32_70b_k10_wooden_pencil,wooden_pencil,input_files\ideas_wooden_pencil.csv,llama3.3:latest,sentence-transformers/all-mpnet-base-v2,10,CoT,20260501,3,True
automobile_tire,neurips_aut_pooled_llama32_70b_k10_automobile_tire,automobile_tire,input_files\ideas_automobile_tire.csv,llama3.3:latest,sentence-transformers/all-mpnet-base-v2,10,CoT,20260501,3,True


## 12. Preflight checks

This cell verifies that the input files exist and have the columns expected by the utility function.

In [12]:
for object_name, exp_arg in experiment_args_by_object.items():
    path = Path(exp_arg["input_csv_path"])
    assert path.exists(), f"Missing input CSV: {path}"

    df = pd.read_csv(path)
    missing = set(required_input_cols) - set(df.columns)
    assert not missing, f"{path} missing columns: {missing}"

    print(
        f"{object_name:16s} | rows={len(df):5d} | "
        f"id unique={df['id'].is_unique} | path={path}"
    )

assert forbidden_path.exists()
display(pd.read_csv(forbidden_path))

shoe             | rows= 3694 | id unique=True | path=input_files\ideas_shoe.csv
button           | rows= 3693 | id unique=True | path=input_files\ideas_button.csv
key              | rows= 3702 | id unique=True | path=input_files\ideas_key.csv
wooden_pencil    | rows= 3703 | id unique=True | path=input_files\ideas_wooden_pencil.csv
automobile_tire  | rows= 3705 | id unique=True | path=input_files\ideas_automobile_tire.csv


,object_name,forbidden_idea
0,shoe,used as footwear
1,button,used to fasten things
2,key,used to open a lock
3,wooden_pencil,used for writing
4,automobile_tire,used on wheel of automobile


## 13. Run one object annotation experiment

Start with one object as a smoke test. I recommend starting with the smallest object file or with `shoe`.

This will create/continue:

- `databases/<experiment_name>/`
- `checkpoints/<experiment_name>/`
- `exports/<experiment_name>/`

In [13]:
# Change this to whichever object you want to run first.
OBJECT_TO_RUN = "shoe"

exp_arg = experiment_args_by_object[OBJECT_TO_RUN]

print("Running experiment:")
print(json.dumps(exp_arg, indent=2))

# Uncomment to run.
run_experiment(exp_arg)

Running experiment:
{
  "experiment_name": "neurips_aut_pooled_llama32_70b_k10_shoe",
  "object_name": "shoe",
  "input_csv_path": "input_files\\ideas_shoe.csv",
  "llm_model": "llama3.3:latest",
  "embedding_model": "sentence-transformers/all-mpnet-base-v2",
  "comparison_k": 10,
  "prompt_method": "CoT",
  "replication_id": 20260501,
  "max_attempts": 3,
  "save_csvs": true
}
3694 ideas remaining for the experiment neurips_aut_pooled_llama32_70b_k10_shoe.
Attempt 1.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\anaconda\lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Raiyan Abdul Baten\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🔍 Annotating Ideas:   0%|                                                                    | 0/3694 [00:00<?…


KeyboardInterrupt



## 14. Run all object annotation experiments

Only run this cell once you are satisfied that the single-object smoke test works.

The utility function is checkpointed, so if the notebook stops midway, rerunning this cell should continue from the remaining unprocessed IDs.

In [15]:
RUN_ALL_OBJECTS = True

if RUN_ALL_OBJECTS:
    for object_name in AUT_OBJECT_ORDER:
        exp_arg = experiment_args_by_object[object_name]

        print("=" * 100)
        print(f"Running object: {object_name}")
        print(json.dumps(exp_arg, indent=2))

        run_experiment(exp_arg)
else:
    print("RUN_ALL_OBJECTS is False. Set to True when ready.")

Running object: shoe
{
  "experiment_name": "neurips_aut_pooled_llama32_70b_k10_shoe",
  "object_name": "shoe",
  "input_csv_path": "input_files\\ideas_shoe.csv",
  "llm_model": "llama3.3:latest",
  "embedding_model": "sentence-transformers/all-mpnet-base-v2",
  "comparison_k": 10,
  "prompt_method": "CoT",
  "replication_id": 20260501,
  "max_attempts": 3,
  "save_csvs": true
}
3626 ideas remaining for the experiment neurips_aut_pooled_llama32_70b_k10_shoe.
Attempt 1.


🔍 Annotating Ideas:   0%|                                                                    | 0/3626 [00:00<?…

All ideas have been annotated for the experiment neurips_aut_pooled_llama32_70b_k10_shoe; Exiting.
📦 Exported annotations and codebook for 'neurips_aut_pooled_llama32_70b_k10_shoe' to 'exports/' folder.



Running object: button
{
  "experiment_name": "neurips_aut_pooled_llama32_70b_k10_button",
  "object_name": "button",
  "input_csv_path": "input_files\\ideas_button.csv",
  "llm_model": "llama3.3:latest",
  "embedding_model": "sentence-transformers/all-mpnet-base-v2",
  "comparison_k": 10,
  "prompt_method": "CoT",
  "replication_id": 20260501,
  "max_attempts": 3,
  "save_csvs": true
}
3693 ideas remaining for the experiment neurips_aut_pooled_llama32_70b_k10_button.
Attempt 1.


🔍 Annotating Ideas:   0%|                                                                    | 0/3693 [00:00<?…

All ideas have been annotated for the experiment neurips_aut_pooled_llama32_70b_k10_button; Exiting.
📦 Exported annotations and codebook for 'neurips_aut_pooled_llama32_70b_k10_button' to 'exports/' folder.



Running object: key
{
  "experiment_name": "neurips_aut_pooled_llama32_70b_k10_key",
  "object_name": "key",
  "input_csv_path": "input_files\\ideas_key.csv",
  "llm_model": "llama3.3:latest",
  "embedding_model": "sentence-transformers/all-mpnet-base-v2",
  "comparison_k": 10,
  "prompt_method": "CoT",
  "replication_id": 20260501,
  "max_attempts": 3,
  "save_csvs": true
}
3702 ideas remaining for the experiment neurips_aut_pooled_llama32_70b_k10_key.
Attempt 1.


...

All ideas have been annotated for the experiment neurips_aut_pooled_llama32_70b_k10_key; Exiting.
📦 Exported annotations and codebook for 'neurips_aut_pooled_llama32_70b_k10_key' to 'exports/' folder.



Running object: wooden_pencil
{
  "experiment_name": "neurips_aut_pooled_llama32_70b_k10_wooden_pencil",
  "object_name": "wooden_pencil",
  "input_csv_path": "input_files\\ideas_wooden_pencil.csv",
  "llm_model": "llama3.3:latest",
  "embedding_model": "sentence-transformers/all-mpnet-base-v2",
  "comparison_k": 10,
  "prompt_method": "CoT",
  "replication_id": 20260501,
  "max_attempts": 3,
  "save_csvs": true
}
3703 ideas remaining for the experiment neurips_aut_pooled_llama32_70b_k10_wooden_pencil.
Attempt 1.


🔍 Annotating Ideas:   0%|                                                                    | 0/3703 [00:00<?…

All ideas have been annotated for the experiment neurips_aut_pooled_llama32_70b_k10_wooden_pencil; Exiting.
📦 Exported annotations and codebook for 'neurips_aut_pooled_llama32_70b_k10_wooden_pencil' to 'exports/' folder.



Running object: automobile_tire
{
  "experiment_name": "neurips_aut_pooled_llama32_70b_k10_automobile_tire",
  "object_name": "automobile_tire",
  "input_csv_path": "input_files\\ideas_automobile_tire.csv",
  "llm_model": "llama3.3:latest",
  "embedding_model": "sentence-transformers/all-mpnet-base-v2",
  "comparison_k": 10,
  "prompt_method": "CoT",
  "replication_id": 20260501,
  "max_attempts": 3,
  "save_csvs": true
}
3705 ideas remaining for the experiment neurips_aut_pooled_llama32_70b_k10_automobile_tire.
Attempt 1.


🔍 Annotating Ideas:   0%|                                                                    | 0/3705 [00:00<?…

All ideas have been annotated for the experiment neurips_aut_pooled_llama32_70b_k10_automobile_tire; Exiting.
📦 Exported annotations and codebook for 'neurips_aut_pooled_llama32_70b_k10_automobile_tire' to 'exports/' folder.





## 15. Export annotations for completed experiments

The original utility exports:

- `exports/<experiment_name>/<experiment_name>_codebook.csv`
- `exports/<experiment_name>/<experiment_name>_annotated_ideas.csv`

If `save_csvs=True`, `run_experiment` should do this automatically after successful completion. This cell lets you export or re-export manually.

In [ ]:
EXPORT_COMPLETED = False

if EXPORT_COMPLETED:
    for object_name, exp_arg in experiment_args_by_object.items():
        experiment_name = exp_arg["experiment_name"]

        print("=" * 100)
        print("Exporting:", experiment_name)

        try:
            export_annotations(experiment_name)
        except Exception as e:
            print(f"Could not export {experiment_name}: {e}")
else:
    print("EXPORT_COMPLETED is False. Set to True when needed.")

## 16. Load exported annotations and merge with pooled metadata

After annotation has completed, this cell collects all object-specific annotation files and joins them to the metadata created earlier.

The output is the main file we will later load into the NeurIPS crowding notebook.

In [16]:
annotation_merged_dfs = []
codebook_dfs = []

for object_name, exp_arg in experiment_args_by_object.items():
    experiment_name = exp_arg["experiment_name"]

    annotated_path = Path("exports") / experiment_name / f"{experiment_name}_annotated_ideas.csv"
    codebook_path = Path("exports") / experiment_name / f"{experiment_name}_codebook.csv"
    metadata_path = METADATA_DIR / f"pooled_metadata_{object_name}.csv"

    if not annotated_path.exists():
        print(f"Skipping {object_name}: missing {annotated_path}")
        continue

    annotated_df = pd.read_csv(annotated_path)
    metadata_df = pd.read_csv(metadata_path)

    # The export file uses idea_ids.
    annotated_df = annotated_df.rename(columns={
        "idea_ids": "id",
        "idea_texts": "idea_content_exported",
        "idea_annotation_ids": "llm_bucket_id",
        "idea_for_user_ids": "for_user_id_exported",
        "idea_object_names": "object_name_exported",
        "idea_reasons": "llm_bucket_reason",
    })

    merged = metadata_df.merge(
        annotated_df,
        on="id",
        how="left",
        validate="one_to_one",
    )

    merged["annotation_experiment_name"] = experiment_name
    merged["llm_bucket_object_name"] = object_name
    merged["llm_bucket_key"] = (
        merged["llm_bucket_object_name"].astype(str)
        + "::"
        + merged["llm_bucket_id"].astype("Int64").astype(str)
    )

    annotation_merged_dfs.append(merged)

    if codebook_path.exists():
        codebook_df = pd.read_csv(codebook_path)
        codebook_df["annotation_experiment_name"] = experiment_name
        codebook_df["object_name"] = object_name
        codebook_df["llm_bucket_key"] = (
            codebook_df["object_name"].astype(str)
            + "::"
            + codebook_df["codebook_ids"].astype("Int64").astype(str)
        )
        codebook_dfs.append(codebook_df)

if annotation_merged_dfs:
    aut_bucket_annotations_all = pd.concat(annotation_merged_dfs, ignore_index=True, sort=False)
else:
    aut_bucket_annotations_all = pd.DataFrame()

if codebook_dfs:
    aut_bucket_codebooks_all = pd.concat(codebook_dfs, ignore_index=True, sort=False)
else:
    aut_bucket_codebooks_all = pd.DataFrame()

print("Merged annotation rows:", aut_bucket_annotations_all.shape)
print("Codebook rows:", aut_bucket_codebooks_all.shape)

display(aut_bucket_annotations_all.head())
display(aut_bucket_codebooks_all.head())

Merged annotation rows: (18497, 28)
Codebook rows: (2851, 5)


,id,pooled_row_uid,pooled_row_idx,source_group,provider,model,analysis_scenario_name,scenario_name,temperature,persona_id,object_name,round_id,for_user_id,participant_id,idea_content,ai_generation_uid,run_idx,request_key,bucket_id,bucket_key,idea_content_exported,llm_bucket_id,for_user_id_exported,object_name_exported,llm_bucket_reason,annotation_experiment_name,llm_bucket_object_name,llm_bucket_key
0,11101,002dc31001de0f43357ac671,6614,ai,openai,gpt-5.4,personality_grid,personality_grid,1.3,introverted__agreeable__conscientious__emotionally_stable__open_to_experience,shoe,1,ai_openai_c4bb8beebb697fcee68c,NaN,Turn the shoe into a discreet doorstop by wedging it under a drafty door.,c4bb8beebb697fcee68c,9.0,b0e51e44125c4f64f1961a26ce3479b0c7ab024996a6fcd2430da22957fae354,NaN,NaN,Turn the shoe into a discreet doorstop by wedging it under a drafty door.,9,ai_openai_c4bb8beebb697fcee68c,shoe,The input idea is a very obviously rephrased version of comparison_idea_ID 9.,neurips_aut_pooled_llama32_70b_k10_shoe,shoe,shoe::9
1,11102,00393c6824564e7d7f86c471,2861,human,human,human,human,human,NaN,NaN,shoe,1,human_92,92.0,Meat tenderizer,NaN,NaN,NaN,884.0,shoe::884,Meat tenderizer,232,human_92,shoe,The input idea is a very obviously rephrased version of comparison_idea_ID 232.,neurips_aut_pooled_llama32_70b_k10_shoe,shoe,shoe::232
2,11103,003c37f0c66a3aa215c858e4,859,human,human,human,human,human,NaN,NaN,shoe,1,human_134,134.0,counter weight for grappling hook,NaN,NaN,NaN,780.0,shoe::780,counter weight for grappling hook,185,human_134,shoe,The input idea is not an obvious rephrasing of any comparison_idea_ID.,neurips_aut_pooled_llama32_70b_k10_shoe,shoe,shoe::185
3,11104,0050830f5db737c8825e2c51,15194,ai,gemini,gemini-2.5-flash,personality_grid,personality_grid,1.3,extroverted__agreeable__unconscientious__emotionally_stable__open_to_experience,shoe,1,ai_gemini_3053c67c6febfec9ecaf,NaN,"Ooh! You know, a shoe could totally be a super fun little planter for a succulent or a tiny herb garden!",3053c67c6febfec9ecaf,2.0,590645df1057cd7f64beab673a5fe9cf7d1f53e27b3e94dfe3b2791995dd4f3e,NaN,NaN,"Ooh! You know, a shoe could totally be a super fun little planter for a succulent or a tiny herb garden!",123,ai_gemini_3053c67c6febfec9ecaf,shoe,The input idea is a very obviously rephrased version of comparison_idea_ID 123.,neurips_aut_pooled_llama32_70b_k10_shoe,shoe,shoe::123
4,11105,006a879f50f8162ee89cba1b,16966,ai,gemini,gemini-2.5-flash,personality_grid,personality_grid,1.3,introverted__antagonistic__conscientious__neurotic__open_to_experience,shoe,1,ai_gemini_63f1e8f2fd86d19ff272,NaN,"A boat shoe could serve as a shallow, weighted anchor for small, decorative pond lilies in a backyard water feature, the lacing providing attachment points for the plant's subm...",63f1e8f2fd86d19ff272,4.0,b2efbb0267041e6c940f2bdd4b2a422fa2590b514b39212e25346332af00564c,NaN,NaN,"A boat shoe could serve as a shallow, weighted anchor for small, decorative pond lilies in a backyard water feature, the lacing providing attachment points for the plant's subm...",192,ai_gemini_63f1e8f2fd86d19ff272,shoe,The input idea is a very obviously rephrased version of comparison_idea_ID 192.,neurips_aut_pooled_llama32_70b_k10_shoe,shoe,shoe::192


,codebook_ids,codebook_descriptions,annotation_experiment_name,object_name,llm_bucket_key
0,1,A shoe can be used as a doorstop to hold a door open at a specific angle.,neurips_aut_pooled_llama32_70b_k10_shoe,shoe,shoe::1
1,2,"A sturdy hiking boot could be repurposed as a unique, rustic planter for a small succulent or herb, adding a quirky touch to a windowsill display.",neurips_aut_pooled_llama32_70b_k10_shoe,shoe,shoe::2
2,3,Turn the shoe into a hidden wall safe by hollowing the sole and mounting it in a closet among real pairs.,neurips_aut_pooled_llama32_70b_k10_shoe,shoe,shoe::3
3,4,As a hand bag,neurips_aut_pooled_llama32_70b_k10_shoe,shoe,shoe::4
4,5,Smash a bug on the wall.,neurips_aut_pooled_llama32_70b_k10_shoe,shoe,shoe::5


## 17. Annotation completeness checks

Every pooled row should have an LLM bucket assignment once the object’s experiment is complete.

In [17]:
if len(aut_bucket_annotations_all) == 0:
    print("No completed annotation exports loaded yet.")
else:
    completeness = (
        aut_bucket_annotations_all
        .groupby(["object_name", "source_group"], dropna=False)
        .agg(
            n_rows=("id", "size"),
            n_annotated=("llm_bucket_id", lambda x: x.notna().sum()),
            n_buckets=("llm_bucket_id", "nunique"),
            n_unique_texts=("idea_content", "nunique"),
        )
        .reset_index()
    )

    completeness["annotation_rate"] = completeness["n_annotated"] / completeness["n_rows"]

    display(completeness)

    incomplete = aut_bucket_annotations_all[aut_bucket_annotations_all["llm_bucket_id"].isna()]
    print("Incomplete rows:", len(incomplete))
    display(incomplete.head())

,object_name,source_group,n_rows,n_annotated,n_buckets,n_unique_texts,annotation_rate
0,automobile_tire,ai,3090,3090,500,2560,1.0
1,automobile_tire,human,615,615,301,575,1.0
2,button,ai,3090,3090,360,2470,1.0
3,button,human,603,603,258,583,1.0
4,key,ai,3090,3090,364,2635,1.0
5,key,human,612,612,273,574,1.0
6,shoe,ai,3090,3090,291,2171,1.0
7,shoe,human,604,604,213,532,1.0
8,wooden_pencil,ai,3090,3090,425,2422,1.0
9,wooden_pencil,human,613,613,306,583,1.0


Incomplete rows: 0


,id,pooled_row_uid,pooled_row_idx,source_group,provider,model,analysis_scenario_name,scenario_name,temperature,persona_id,object_name,round_id,for_user_id,participant_id,idea_content,ai_generation_uid,run_idx,request_key,bucket_id,bucket_key,idea_content_exported,llm_bucket_id,for_user_id_exported,object_name_exported,llm_bucket_reason,annotation_experiment_name,llm_bucket_object_name,llm_bucket_key


## 18. Save merged annotation outputs for the crowding notebook

These files will be loaded later to compute the AUT bucket-crowding kernel:

$K^{bucket}_k(x,y)=I\{b_k(x)=b_k(y)\}$

where bucket IDs are object-specific.

In [18]:
if len(aut_bucket_annotations_all) > 0:
    annotations_pkl = AUT_BUCKET_DIR / "aut_pooled_llm_bucket_annotations_all.pkl"
    annotations_csv = AUT_BUCKET_DIR / "aut_pooled_llm_bucket_annotations_all.csv"

    codebooks_pkl = AUT_BUCKET_DIR / "aut_pooled_llm_bucket_codebooks_all.pkl"
    codebooks_csv = AUT_BUCKET_DIR / "aut_pooled_llm_bucket_codebooks_all.csv"

    aut_bucket_annotations_all.to_pickle(annotations_pkl)
    aut_bucket_annotations_all.to_csv(annotations_csv, index=False)

    aut_bucket_codebooks_all.to_pickle(codebooks_pkl)
    aut_bucket_codebooks_all.to_csv(codebooks_csv, index=False)

    print("Saved:")
    print(annotations_pkl)
    print(annotations_csv)
    print(codebooks_pkl)
    print(codebooks_csv)
else:
    print("No annotations to save yet.")

Saved:
analysis_outputs\aut_bucket_annotation\aut_pooled_llm_bucket_annotations_all.pkl
analysis_outputs\aut_bucket_annotation\aut_pooled_llm_bucket_annotations_all.csv
analysis_outputs\aut_bucket_annotation\aut_pooled_llm_bucket_codebooks_all.pkl
analysis_outputs\aut_bucket_annotation\aut_pooled_llm_bucket_codebooks_all.csv


## 19. Optional: copy exports into analysis output directory

This makes the EMNLP utility exports easier to archive with the NeurIPS project outputs.

In [19]:
COPY_EXPORTS = False

if COPY_EXPORTS:
    for object_name, exp_arg in experiment_args_by_object.items():
        experiment_name = exp_arg["experiment_name"]
        src_dir = Path("exports") / experiment_name
        dst_dir = EXPORT_COPY_DIR / experiment_name

        if src_dir.exists():
            if dst_dir.exists():
                shutil.rmtree(dst_dir)
            shutil.copytree(src_dir, dst_dir)
            print(f"Copied {src_dir} -> {dst_dir}")
        else:
            print(f"Missing export directory: {src_dir}")
else:
    print("COPY_EXPORTS is False. Set to True when needed.")

COPY_EXPORTS is False. Set to True when needed.


## 20. Final status report

In [20]:
print("=" * 100)
print("POOLED INPUT FILES")
print("=" * 100)

for object_name in AUT_OBJECT_ORDER:
    input_path = INPUT_FILES_DIR / f"ideas_{object_name}.csv"
    metadata_path = METADATA_DIR / f"pooled_metadata_{object_name}.csv"

    if input_path.exists():
        df = pd.read_csv(input_path)
        print(f"{object_name:16s} | input rows={len(df):5d} | {input_path}")

    if metadata_path.exists():
        meta = pd.read_csv(metadata_path)
        print(f"{'':16s} | metadata rows={len(meta):5d} | {metadata_path}")

print("\n" + "=" * 100)
print("EXPERIMENTS")
print("=" * 100)

for object_name, exp_arg in experiment_args_by_object.items():
    experiment_name = exp_arg["experiment_name"]
    annotated_path = Path("exports") / experiment_name / f"{experiment_name}_annotated_ideas.csv"
    codebook_path = Path("exports") / experiment_name / f"{experiment_name}_codebook.csv"

    print(f"{object_name:16s} | {experiment_name}")
    print(f"{'':16s} | annotated exists: {annotated_path.exists()} | {annotated_path}")
    print(f"{'':16s} | codebook exists:   {codebook_path.exists()} | {codebook_path}")

POOLED INPUT FILES
shoe             | input rows= 3694 | input_files\ideas_shoe.csv
                 | metadata rows= 3694 | analysis_outputs\aut_bucket_annotation\metadata\pooled_metadata_shoe.csv
button           | input rows= 3693 | input_files\ideas_button.csv
                 | metadata rows= 3693 | analysis_outputs\aut_bucket_annotation\metadata\pooled_metadata_button.csv
key              | input rows= 3702 | input_files\ideas_key.csv
                 | metadata rows= 3702 | analysis_outputs\aut_bucket_annotation\metadata\pooled_metadata_key.csv
wooden_pencil    | input rows= 3703 | input_files\ideas_wooden_pencil.csv
                 | metadata rows= 3703 | analysis_outputs\aut_bucket_annotation\metadata\pooled_metadata_wooden_pencil.csv
automobile_tire  | input rows= 3705 | input_files\ideas_automobile_tire.csv
                 | metadata rows= 3705 | analysis_outputs\aut_bucket_annotation\metadata\pooled_metadata_automobile_tire.csv

EXPERIMENTS
shoe             | neurips_aut_